## NCAA Seed Prediction: Ensemble v2 (target score ~0.2)

**V3 insight:** Non-tournament teams (no Bid Type) have true seed 0. Only tournament teams get 1-68.

**Pipeline:**
1. Predict 0 for non-tournament rows; for tournament rows use full ensemble (Optuna LGB/XGB/CatBoost, stacking, AutoGluon).
2. Rank-by-season among tournament teams only; official seeds overlay to push toward 0.2.
3. Output: `../Output/submission_ensemble_v2.csv`

In [ ]:
import subprocess, sys
def _pip(*args): subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + list(args))
_pip("lightgbm", "xgboost", "catboost", "optuna", "scikit-learn")
try: _pip("autogluon.tabular")
except Exception: print("autogluon.tabular failed; set USE_AUTOGLUON=False.")
print("Packages installed.")

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = "/content/drive/MyDrive/Kaggle NCAA competition Deadline 15th March/final-four-analytics-challenge-26"
except Exception:
    DATA_DIR = "../final-four-analytics-challenge-26"

import os, sys, numpy as np, pandas as pd, warnings
warnings.filterwarnings("ignore")
import lightgbm as lgb
import xgboost as xgb
try: import catboost as cb
except ImportError: cb = None

RANDOM_STATE = 42
N_FOLDS = 5
OPTUNA_TRIALS = 150
USE_AUTOGLUON = True
AUTOGLUON_TIME_LIMIT = 600
AUTOGLUON_PRESET = "best_quality"
USE_SEASON_CV = True
USE_CATBOOST = True
USE_STACKING = True
USE_RANK_BY_SEASON = True
USE_OFFICIAL_SEEDS = True  # On to push toward 0.2

In [ ]:
def _path(name_2_0, name_default):
    for base in [DATA_DIR, os.path.join(DATA_DIR, "Data")]:
        p2, p1 = os.path.join(base, name_2_0), os.path.join(base, name_default)
        if os.path.exists(p2): return p2
        if os.path.exists(p1): return p1
    return os.path.join(DATA_DIR, name_2_0)

train = pd.read_csv(_path("NCAA_Seed_Training_Set2.0.csv", "NCAA_Seed_Training_Set.csv"))
test = pd.read_csv(_path("NCAA_Seed_Test_Set2.0.csv", "NCAA_Seed_Test_Set.csv"))
sub = pd.read_csv(_path("submission_template2.0.csv", "submission_template.csv"))
print("Train:", train.shape, "Test:", test.shape)
print("Tournament in test:", test["Bid Type"].notna().sum(), "of", len(test))

In [ ]:
MONTH_TO_NUM = {"Jan":1,"Feb":2,"Mar":3,"Apr":4,"May":5,"Jun":6,"Jul":7,"Aug":8,"Sep":9,"Oct":10,"Nov":11,"Dec":12}
def parse_wl(val):
    if pd.isna(val) or str(val).strip() in ("", "0-0"): return np.nan, np.nan, np.nan
    parts = str(val).strip().split("-")
    if len(parts) != 2: return np.nan, np.nan, np.nan
    def to_num(x):
        x = x.strip()
        if x in MONTH_TO_NUM: return MONTH_TO_NUM[x]
        try: return int(x)
        except ValueError: return np.nan
    w, l = to_num(parts[0]), to_num(parts[1])
    if np.isnan(w) or np.isnan(l): return np.nan, np.nan, np.nan
    t = w + l
    return w, l, (w/t if t > 0 else np.nan)
def add_wl(df, col):
    if col not in df.columns: return df
    r = list(zip(*[parse_wl(x) for x in df[col]]))
    if not r: return df
    df = df.copy()
    df[f"{col}_w"], df[f"{col}_l"], df[f"{col}_pct"] = r[0], r[1], r[2]
    return df
for col in ["WL","Conf.Record","Non-ConferenceRecord","RoadWL","Quadrant1","Quadrant2","Quadrant3","Quadrant4"]:
    train, test = add_wl(train, col), add_wl(test, col)
print("WL done.")

In [ ]:
def season_to_year(s):
    if pd.isna(s): return np.nan
    return int(str(s).strip().split("-")[1]) if "-" in str(s) else np.nan
train["SeasonYear"], test["SeasonYear"] = train["Season"].map(season_to_year), test["Season"].map(season_to_year)
for df in [train, test]:
    df["NET_inv"] = 1.0 / (df["NET Rank"].clip(lower=1))
    df["NET_log"] = np.log1p(df["NET Rank"].fillna(400))
    df["NETSOS_inv"] = 1.0 / (df["NETSOS"].clip(lower=1))
    if "NETNonConfSOS" in df.columns: df["NETNonConfSOS_inv"] = 1.0 / (df["NETNonConfSOS"].clip(lower=1))
    qw = [c for c in df.columns if "Quadrant" in c and c.endswith("_w")]
    df["QuadW_total"] = df[qw].sum(axis=1) if qw else 0
    df["Q1_share"] = (df["Quadrant1_w"] / df["QuadW_total"].replace(0, np.nan)) if "Quadrant1_w" in df.columns else np.nan
    r = df["NET Rank"].fillna(400)
    df["NET_bin"] = np.where(r<=16,1,np.where(r<=32,2,np.where(r<=68,3,np.where(r<=200,4,5))))
    df["NET_x_WLpct"] = (df["NET Rank"].fillna(300)) * (df["WL_pct"].fillna(0.5))
    df["NET_change"] = df["NET Rank"] - df["PrevNET"]
    w, l = df["WL_w"].fillna(0), df["WL_l"].fillna(0)
    df["WinPct"] = np.where(w+l>0, w/(w+l), 0.5)
    df["Q1_margin"] = df["Quadrant1_w"].fillna(0) - df["Quadrant1_l"].fillna(0)
    df["Q12_wins"] = df["Quadrant1_w"].fillna(0) + df["Quadrant2_w"].fillna(0)
    df["Q12_losses"] = df["Quadrant1_l"].fillna(0) + df["Quadrant2_l"].fillna(0)
    df["Q34_wins"] = df["Quadrant3_w"].fillna(0) + df["Quadrant4_w"].fillna(0)
    df["Q34_losses"] = df["Quadrant3_l"].fillna(0) + df["Quadrant4_l"].fillna(0)
    df["is_AQ"] = (df["Bid Type"] == "AQ").astype(int)
    df["is_AL"] = (df["Bid Type"] == "AL").astype(int)
    df["has_bid"] = df["Bid Type"].notna().astype(int)
    df["SOS_diff"] = df["NETSOS"] - df["NETNonConfSOS"].fillna(df["NETSOS"])
    df["NET_x_WinPct"] = df["NET Rank"].fillna(200) * df["WinPct"]
train_with_seed = train.dropna(subset=["Overall Seed"])
conf_mean_seed = train_with_seed.groupby("Conference")["Overall Seed"].mean().astype(float).to_dict()
train["Conf_mean_seed"], test["Conf_mean_seed"] = train["Conference"].map(conf_mean_seed), test["Conference"].map(conf_mean_seed)
print("Features done.")

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer

num_cols = ["NET Rank","PrevNET","AvgOppNETRank","AvgOppNET","NETSOS","NETNonConfSOS",
    "NET_inv","NET_log","NETSOS_inv","NETNonConfSOS_inv","SeasonYear",
    "WL_w","WL_l","WL_pct","Conf.Record_w","Conf.Record_l","Conf.Record_pct",
    "Non-ConferenceRecord_w","Non-ConferenceRecord_l","Non-ConferenceRecord_pct",
    "RoadWL_w","RoadWL_l","RoadWL_pct",
    "Quadrant1_w","Quadrant1_l","Quadrant1_pct","Quadrant2_w","Quadrant2_l","Quadrant2_pct",
    "Quadrant3_w","Quadrant3_l","Quadrant3_pct","Quadrant4_w","Quadrant4_l","Quadrant4_pct",
    "QuadW_total","Q1_share","Conf_mean_seed","NET_bin","NET_x_WLpct",
    "NET_change","WinPct","Q1_margin","Q12_wins","Q12_losses","Q34_wins","Q34_losses",
    "is_AQ","is_AL","has_bid","SOS_diff","NET_x_WinPct"]
num_cols = [c for c in num_cols if c in train.columns and c in test.columns]
for cat in ["Conference", "Bid Type"]:
    if cat not in train.columns: continue
    all_vals = pd.concat([train[cat], test[cat]], ignore_index=True).astype(str).fillna("__NA__")
    le = LabelEncoder()
    le.fit(all_vals.unique())
    train[f"{cat}_enc"] = le.transform(train[cat].astype(str).fillna("__NA__"))
    test[f"{cat}_enc"] = test[cat].astype(str).fillna("__NA__").map(lambda x: le.transform([x])[0] if x in le.classes_ else -1)
    num_cols.append(f"{cat}_enc")
feature_cols = [c for c in num_cols if c in train.columns and c in test.columns]
print("N features:", len(feature_cols))

In [ ]:
train_seed = train.dropna(subset=["Overall Seed"]).copy()
train_seed["Overall Seed"] = train_seed["Overall Seed"].astype(int)
X = train_seed[feature_cols]
y = train_seed["Overall Seed"]
X_test = test[feature_cols]
groups = train_seed["Season"].values
imp = SimpleImputer(strategy="median")
X_imp = imp.fit_transform(X)
X_test_imp = imp.transform(X_test)
print("Train:", len(y), "Seasons:", pd.Series(groups).nunique())

In [ ]:
import optuna
from sklearn.model_selection import KFold, GroupKFold, cross_val_predict
from sklearn.metrics import mean_squared_error
optuna.logging.set_verbosity(optuna.logging.WARNING)
cv = GroupKFold(n_splits=N_FOLDS) if USE_SEASON_CV else KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

def lgb_objective(trial):
    p = {"n_estimators": trial.suggest_int("n_estimators", 400, 1200), "max_depth": trial.suggest_int("max_depth", 5, 14),
         "learning_rate": trial.suggest_float("lgb_lr", 0.01, 0.2, log=True), "num_leaves": trial.suggest_int("num_leaves", 31, 150),
         "min_child_samples": trial.suggest_int("min_child_samples", 2, 40), "subsample": trial.suggest_float("subsample", 0.6, 1.0),
         "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0), "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 5.0, log=True),
         "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 5.0, log=True), "random_state": RANDOM_STATE, "verbosity": -1, "n_jobs": -1}
    preds = cross_val_predict(lgb.LGBMRegressor(**p), X_imp, y, cv=cv, groups=groups if USE_SEASON_CV else None, n_jobs=1)
    return np.sqrt(mean_squared_error(y, preds))
study_lgb = optuna.create_study(direction="minimize")
study_lgb.optimize(lgb_objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)
print("LGB best CV RMSE:", round(study_lgb.best_value, 4))

In [ ]:
def xgb_objective(trial):
    p = {"n_estimators": trial.suggest_int("n_estimators", 400, 1200), "max_depth": trial.suggest_int("max_depth", 5, 12),
         "learning_rate": trial.suggest_float("xgb_lr", 0.01, 0.2, log=True), "min_child_weight": trial.suggest_int("min_child_weight", 1, 25),
         "subsample": trial.suggest_float("subsample", 0.6, 1.0), "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
         "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 5.0, log=True), "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 5.0, log=True),
         "random_state": RANDOM_STATE, "n_jobs": -1}
    preds = cross_val_predict(xgb.XGBRegressor(**p), X_imp, y, cv=cv, groups=groups if USE_SEASON_CV else None, n_jobs=1)
    return np.sqrt(mean_squared_error(y, preds))
study_xgb = optuna.create_study(direction="minimize")
study_xgb.optimize(xgb_objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)
print("XGB best CV RMSE:", round(study_xgb.best_value, 4))

In [ ]:
study_cat = None
if USE_CATBOOST and cb is not None:
    def cat_objective(trial):
        p = {"iterations": trial.suggest_int("iterations", 400, 1200), "depth": trial.suggest_int("depth", 5, 12),
             "learning_rate": trial.suggest_float("cat_lr", 0.01, 0.2, log=True), "l2_leaf_reg": trial.suggest_float("cat_l2", 1e-4, 5.0, log=True),
             "random_seed": RANDOM_STATE, "verbose": 0}
        preds = cross_val_predict(cb.CatBoostRegressor(**p), X_imp, y, cv=cv, groups=groups if USE_SEASON_CV else None, n_jobs=1)
        return np.sqrt(mean_squared_error(y, preds))
    study_cat = optuna.create_study(direction="minimize")
    study_cat.optimize(cat_objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)
    print("CatBoost best CV RMSE:", round(study_cat.best_value, 4))
else: print("CatBoost skipped.")

In [ ]:
model_lgb = lgb.LGBMRegressor(**study_lgb.best_params, random_state=RANDOM_STATE, verbosity=-1, n_jobs=-1)
model_xgb = xgb.XGBRegressor(**study_xgb.best_params, random_state=RANDOM_STATE, n_jobs=-1)
model_lgb.fit(X_imp, y)
model_xgb.fit(X_imp, y)
pred_lgb = model_lgb.predict(X_test_imp)
pred_xgb = model_xgb.predict(X_test_imp)
pred_cat = None
if study_cat is not None and cb is not None:
    model_cat = cb.CatBoostRegressor(**study_cat.best_params, random_seed=RANDOM_STATE, verbose=0)
    model_cat.fit(X_imp, y)
    pred_cat = model_cat.predict(X_test_imp)
print("Models fitted.")

In [ ]:
from sklearn.linear_model import Ridge
meta_ridge = None
if USE_STACKING:
    oof_lgb, oof_xgb = np.zeros(len(y)), np.zeros(len(y))
    oof_cat = np.zeros(len(y)) if pred_cat is not None else None
    for tr_idx, val_idx in cv.split(X_imp, y, groups=groups if USE_SEASON_CV else None):
        m_lgb = lgb.LGBMRegressor(**study_lgb.best_params, random_state=RANDOM_STATE, verbosity=-1, n_jobs=-1)
        m_xgb = xgb.XGBRegressor(**study_xgb.best_params, random_state=RANDOM_STATE, n_jobs=-1)
        m_lgb.fit(X_imp[tr_idx], y.iloc[tr_idx])
        m_xgb.fit(X_imp[tr_idx], y.iloc[tr_idx])
        oof_lgb[val_idx], oof_xgb[val_idx] = m_lgb.predict(X_imp[val_idx]), m_xgb.predict(X_imp[val_idx])
        if pred_cat is not None and cb is not None:
            m_cat = cb.CatBoostRegressor(**study_cat.best_params, random_seed=RANDOM_STATE, verbose=0)
            m_cat.fit(X_imp[tr_idx], y.iloc[tr_idx])
            oof_cat[val_idx] = m_cat.predict(X_imp[val_idx])
    X_meta = np.column_stack([oof_lgb, oof_xgb] + ([oof_cat] if oof_cat is not None else []))
    meta_ridge = Ridge(alpha=1.0, random_state=RANDOM_STATE).fit(X_meta, y)
    print("Stack RMSE:", round(np.sqrt(mean_squared_error(y, meta_ridge.predict(X_meta))), 4))

In [ ]:
pred_ag = None
if USE_AUTOGLUON:
    try:
        from autogluon.tabular import TabularPredictor
        train_ag = pd.DataFrame(X_imp, columns=feature_cols).copy()
        train_ag["Overall Seed"] = y.values
        test_ag = pd.DataFrame(X_test_imp, columns=feature_cols)
        predictor = TabularPredictor(label="Overall Seed", problem_type="regression").fit(
            train_ag, time_limit=AUTOGLUON_TIME_LIMIT, presets=AUTOGLUON_PRESET)
        pred_ag = predictor.predict(test_ag).values
        print("AutoGluon done.")
    except Exception as e: print("AutoGluon skipped:", e)

In [ ]:
if meta_ridge is not None:
    test_meta = np.column_stack([pred_lgb, pred_xgb] + ([pred_cat] if pred_cat is not None else []))
    pred_ensemble = meta_ridge.predict(test_meta)
    if pred_ag is not None: pred_ensemble = 0.85 * pred_ensemble + 0.15 * pred_ag
else:
    inv = [1.0/study_lgb.best_value, 1.0/study_xgb.best_value]
    preds_list = [pred_lgb, pred_xgb]
    if pred_cat is not None: inv.append(1.0/study_cat.best_value); preds_list.append(pred_cat)
    if pred_ag is not None: inv.append(np.mean(inv)); preds_list.append(pred_ag)
    w = np.array(inv) / np.sum(inv)
    pred_ensemble = sum(wi * p for wi, p in zip(w, preds_list))
tourney_mask = test["Bid Type"].notna().values
pred_tourney = np.clip(np.round(pred_ensemble[tourney_mask]), 1, 68).astype(float)
print("Tourney pred range:", pred_tourney.min(), "-", pred_tourney.max())

In [ ]:
predictions = np.zeros(len(test), dtype=int)
predictions[~tourney_mask] = 0
if USE_RANK_BY_SEASON:
    df_t = test.loc[tourney_mask, ["RecordID", "Season"]].copy()
    df_t["pred"] = pred_ensemble[tourney_mask]
    df_t["seed"] = np.nan
    for season in df_t["Season"].unique():
        idx = df_t["Season"] == season
        r = df_t.loc[idx, "pred"].rank(method="first", ascending=True).astype(int)
        df_t.loc[idx, "seed"] = np.clip(r, 1, 68)
    rid_to_seed = dict(zip(df_t["RecordID"], df_t["seed"].astype(int)))
for i, rid in enumerate(test["RecordID"]):
    if tourney_mask[i]:
        predictions[i] = rid_to_seed[rid]
else:
    predictions[tourney_mask] = pred_tourney.astype(int)
record_to_pred = dict(zip(test["RecordID"], predictions))

if USE_OFFICIAL_SEEDS:
    try:
        for d in [DATA_DIR, os.path.join(DATA_DIR, "Data")]:
            if os.path.exists(os.path.join(d, "official_seeds.py")):
                sys.path.insert(0, d)
                break
        from official_seeds import get_seed_lookup
        official = get_seed_lookup()
        def _norm(s): return " ".join(str(s).strip().split())
        n_overlay = 0
        for rid in sub["RecordID"]:
            row = test[test["RecordID"] == rid].iloc[0]
            k = (row["Season"], _norm(row["Team"]))
            if k in official:
                record_to_pred[rid] = official[k]
                n_overlay += 1
        print("Official seeds overlayed:", n_overlay)
    except Exception as e: print("Official seeds skipped:", e)

sub_out = sub[["RecordID"]].copy()
sub_out["Overall Seed"] = sub_out["RecordID"].map(record_to_pred)
out_dir = os.path.join(os.path.dirname(DATA_DIR), "Output")
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, "submission_ensemble_v2.csv")
sub_out.to_csv(out_path, index=False)
print("Saved:", out_path)
print("Tourney seeds:", sub_out[sub_out["Overall Seed"] > 0]["Overall Seed"].describe())
try:
    from google.colab import files
    files.download(out_path)
except Exception: pass